# 00 - Conversão dos dados de origem (CSV -> pickle)

Converte os dois CSV do Kaggle em pickles, para as sessões seguintes abrirem os dados muito mais rápidamente

**O que este notebook faz**
1. Converte `accepted_2007_to_2018Q4.csv.gz` e `rejected_2007_to_2018Q4.csv.gz` (pasta `data/raw/`) em pickles (pasta `data/processed/`).
2. Verifica a cópia: número de linhas e colunas, tipos, formato dos valores.
3. Regista os números em `docs/dados_origem.md`, para o relatório.

**Decisões** (explicadas também em `src/lending/conversion.py`)
- Todas as colunas são lidas como **texto**: a conversão muda só o formato do ficheiro, não interpreta valores. Tipos e limpeza ficam para a Fase 2 (ELT).
- Só o **campo vazio** conta como valor em falta; textos como `"n/a"` ficam como estão.

**O que NÃO se faz aqui:** limpeza, exclusões, EDA. Este notebook não tira conclusões dos dados.

**Antes de correr:** a conversão do ficheiro *accepted* precisa de vários GB de RAM livre, por isso recomendamos fechar programas pesados.


In [ ]:
# Setup: imports e caminhos do projeto
import gc
import platform
import sys
import time
from pathlib import Path

# Importar o pacote do projeto.
try:
    import lending
except ModuleNotFoundError:
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / "src" / "lending").is_dir():
            sys.path.insert(0, str(parent / "src"))
            break
    import lending

import numpy as np
import pandas as pd
import pyarrow

from lending import config
from lending.conversion import CsvToPickleConverter, DatasetSummary
from lending.storage import PickleStore

# Mostrar todas as colunas quando se imprime um DataFrame
pd.set_option("display.max_columns", None)

print(f"Python {platform.python_version()} | pandas {pd.__version__} | "
      f"pyarrow {pyarrow.__version__} | numpy {np.__version__}")
print("Raiz do projeto:", config.PROJECT_ROOT)


## 1. Conversão

Cada conversão corre **uma única vez**. Se o pickle já existir é saltada (se quiser refazer basta colocar este parametro no convert: `convert(overwrite=True)`).

- *accepted*: 151 colunas -> blocos de 100 000 linhas.
- *rejected*: 9 colunas -> blocos de 1 000 000 linhas.

Se der `MemoryError` ou o computador ficar muito lento, basta reduzir o `chunksize`.


In [ ]:
accepted_converter = CsvToPickleConverter(
    csv_path=config.ACCEPTED_CSV,
    pickle_path=config.ACCEPTED_RAW_PKL,
    chunksize=100_000,
)
accepted_converter.convert()

# Libertar a memória usada durante a conversão antes da próxima
_ = gc.collect()


In [ ]:
rejected_converter = CsvToPickleConverter(
    csv_path=config.REJECTED_CSV,
    pickle_path=config.REJECTED_RAW_PKL,
    chunksize=1_000_000,
)
rejected_converter.convert()

_ = gc.collect()


## 2. Verificação

Para cada pickle:
1. Abrir e medir o tempo (o esperado é que seja em segundos).
2. Confirmar o número de colunas esperado (151 e 9) e que todas são texto.
3. Ver o **formato** dos valores numa amostra aleatória com semente.
   *Não é uma amostra para tirar conclusões*, serve só para planear o parsing da Etapa 2
   É aleatória porque o enunciado proíbe usar as primeiras linhas como amostra
4. Ver as últimas linhas do ficheiro: é comum ficheiros exportados terminarem com linhas de totais ou rodapé. Se existirem ficam na cópia e saem na limpeza (Etapa 2).

Os ficheiros são verificados um de cada vez, libertando a memória entre eles.


In [ ]:
# accepted: abrir e medir
t0 = time.perf_counter()
accepted = PickleStore.load(config.ACCEPTED_RAW_PKL, expected_type=pd.DataFrame)
load_seconds = time.perf_counter() - t0

accepted_summary = DatasetSummary.from_dataframe(
    "accepted", accepted, config.ACCEPTED_CSV, config.ACCEPTED_RAW_PKL, load_seconds
)
print(accepted_summary)

# verificações
assert accepted.shape[1] == config.EXPECTED_COLUMNS["accepted"], "Número de colunas inesperado"
assert (accepted.dtypes == accepted.dtypes.iloc[0]).all(), "Há colunas que não são texto"
print("OK: colunas e tipos como esperado |", accepted.dtypes.iloc[0])


In [ ]:
# Formato dos valores (amostra aleatória com semente - não serve para conclusões)
format_columns = ["id", "issue_d", "loan_amnt", "term", "int_rate", "grade",
                  "emp_length", "annual_inc", "dti", "revol_util",
                  "earliest_cr_line", "loan_status"]
format_columns = [c for c in format_columns if c in accepted.columns]

accepted[format_columns].sample(5, random_state=config.RANDOM_SEED)


In [ ]:
# Fim do ficheiro
accepted[["id", "loan_amnt", "issue_d", "loan_status"]].tail(3)


In [ ]:
# Libertar memória antes de abrir o próximo ficheiro
del accepted
_ = gc.collect()


In [ ]:
# rejected: abrir e medir
t0 = time.perf_counter()
rejected = PickleStore.load(config.REJECTED_RAW_PKL, expected_type=pd.DataFrame)
load_seconds = time.perf_counter() - t0

rejected_summary = DatasetSummary.from_dataframe(
    "rejected", rejected, config.REJECTED_CSV, config.REJECTED_RAW_PKL, load_seconds
)
print(rejected_summary)

assert rejected.shape[1] == config.EXPECTED_COLUMNS["rejected"], "Número de colunas inesperado"
assert (rejected.dtypes == rejected.dtypes.iloc[0]).all(), "Há colunas que não são texto"
print("OK: colunas e tipos como esperado |", rejected.dtypes.iloc[0])
print("Colunas:", rejected.columns.tolist())


In [ ]:
# Formato dos valores (amostra aleatória com semente) e fim do ficheiro
display(rejected.sample(5, random_state=config.RANDOM_SEED))
display(rejected.tail(3))


In [ ]:
del rejected
_ = gc.collect()


## 3. Registo para o relatório

Escreve `docs/dados_origem.md` com a tabela dos dois ficheiros e as versões usadas.

Preenche a data em que foi descarregada os dados do Kaggle.


In [ ]:
DOWNLOAD_DATE = "2026-10-14"   # data do download do Kaggle

summaries = [accepted_summary, rejected_summary]
out = DatasetSummary.write_markdown(summaries, config.DOCS_DIR / "dados_origem.md",
                                    download_date=DOWNLOAD_DATE)
print("Escrito:", out.relative_to(config.PROJECT_ROOT))

pd.DataFrame([s.as_dict() for s in summaries]).set_index("name").round(1)
